In [ ]:
!pip install -U unsloth trl transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
except Exception as e:
    print(f"Login failed: {e}")
    print("\nFIX: Go to the Secrets tab (key icon), add 'HF_TOKEN', and enable 'Notebook access'.")

Successfully logged into Hugging Face Hub!


In [ ]:
import torch
import wandb
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
max_seq_length= 2048
model, tokenizer = FastLanguageModel.from_pretrained(model_name= "unsloth/llama-3-8b-bnb-4bit",
                                                     max_seq_length= max_seq_length,
                                                     dtype= torch.float16,
                                                     load_in_4bit= True,)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
model= FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules= ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha= 32,
    lora_dropout= 0,
    bias= "none",
    use_gradient_checkpointing= "unsloth",
    random_state=42)

In [ ]:
dataset_id = "FathyElghoneimy/alpaca-code-generation-curated"
dataset = load_dataset(dataset_id)

README.md:   0%|          | 0.00/595 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/338k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/50.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/47.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/640 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/80 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
def format_instruction(example):
  if example.get("input", "") !="":
    text=f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
  else:
    text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

  return {"text": text + tokenizer.eos_token}


In [ ]:
dataset = dataset.map(format_instruction)

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
Traning_args = TrainingArguments(
    per_device_train_batch_size= 2,
    gradient_accumulation_steps=4,
    warmup_steps= 10,
    num_train_epochs= 4,
    learning_rate= 2e-4,
    fp16=True,
    bf16=False,
    logging_steps= 10,
    eval_steps= 0,         # Disable evaluation steps to prevent associated saves
    eval_strategy="no",   # Explicitly disable evaluation strategy
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    output_dir="outputs",
    save_strategy="no", # Disable saving during training
    save_steps=0,       # Explicitly prevent saving training arguments
    push_to_hub=False,  # Disable pushing to hub during training
    report_to="none"    # Prevent reporting tools from saving state
)

In [ ]:
trainer = SFTTrainer(
    model= model,
    tokenizer= tokenizer,
    train_dataset= dataset["train"],
    eval_dataset= dataset["validation"],
    args= Traning_args,
    dataset_text_field="text",
    max_seq_length= max_seq_length
)

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/640 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 640 | Num Epochs = 4 | Total steps = 320
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,0.270990
20,0.309186
30,0.297183
40,0.319379
50,0.328139
60,0.367259
70,0.328683
80,0.389411
90,0.288745
100,0.295541


TrainOutput(global_step=320, training_loss=0.22163236290216445, metrics={'train_runtime': 917.6074, 'train_samples_per_second': 2.79, 'train_steps_per_second': 0.349, 'total_flos': 1.3717850877689856e+16, 'train_loss': 0.22163236290216445, 'epoch': 4.0})

In [ ]:
model.push_to_hub("FathyElghoneimy/llama-3-8b-lora-code-gen", token=hf_token)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            

Saved model to https://huggingface.co/FathyElghoneimy/llama-3-8b-lora-code-gen
